In [89]:
import pandas as pd
import json
import seaborn as sns
import matplotlib.pyplot as mpl

In [90]:
subreddits = [
    # main subreddits
    "soccer",
    "premierleague",
    "football",
    "theother14",

    # big clubs
    "reddevils",    # man u
    "LiverpoolFC",
    "gunners",      # arsenal
    "chelseafc",
    "MCFC",         # man city
    "coys",         # spurs
    
    # others
    "FantasyPL",
    "Championship",
    
    # diversity subreddits
    "realmadrid",
    "Barca",
    "fcbayern",
    "psg",
    "ACMilan",
]

get and display raw information

In [91]:
df = pd.read_csv("../reddit-scrape-data/data.csv", encoding="utf-8")
df

,author,author_flair,title,content,post_id,date,upvotes,upvote_ratio,num_comments,comments,subreddit
0,Sparky-moon,:transpride::AS_Roma:,The Congolese courts have sentenced Jean-Guy B...,"On Tuesday, March 10, 2026, the Congolese cour...",1rpxyqy,1.773152e+09,5,1.00,1,['Some old rich fuck getting put away for misu...,soccer
1,OkayFine101,:Manchester_United:,[@WestHam] West Ham United's dressing room cel...,NaN,1rpxx4c,1.773152e+09,8,1.00,2,[],soccer
2,SolitasTT,:Liverpool:,Eni Aluko wins Joey Barton libel case over pos...,NaN,1rpxsqj,1.773151e+09,11,0.87,5,['Not a good day for Joey Barton. Hopefully ma...,soccer
3,Sparky-moon,:transpride::AS_Roma:,FIFA has cut over $100 million from its operat...,The people who raised the matter of the budget...,1rpxspb,1.773151e+09,3,1.00,1,"[""It wouldn't surprise me if there a comes a d...",soccer
4,AutoModerator,NaN,Daily Discussion,# Welcome to the [r/soccer](https://old.reddit...,1rpxqc2,1.773151e+09,5,1.00,7,['I will riot if we don’t get ucl football nex...,soccer
...,...,...,...,...,...,...,...,...,...,...,...
5095,Claija79,Bot Mexicano 🇲🇽,[Catanzaro] 18th min. Straight away a forced c...,NaN,1qypegy,1.770498e+09,56,0.96,17,"['We’re cursed', 'Lmao. Lmfao even.', 'AC Infi...",ACMilan
5096,Claija79,Bot Mexicano 🇲🇽,El Bebote on IG: “I miss you… but almost there...,NaN,1qyp8yd,1.770498e+09,311,0.98,40,"['6/7 days a week I forget he plays for us', '...",ACMilan
5097,Claija79,Bot Mexicano 🇲🇽,Kevin Zeroli scored for Juve Stabia against Pa...,NaN,1qymv5x,1.770492e+09,77,0.99,4,['https://preview.redd.it/wtj5orbri4ig1.jpeg?w...,ACMilan
5098,Claija79,Bot Mexicano 🇲🇽,Genoa [2] - 2 Napoli - Lorenzo Colombo 57',NaN,1qylee9,1.770489e+09,102,1.00,12,['Good depth and good for lists. \n\nNgl I onl...,ACMilan


filter out all those that are before the latest season (1 aug 2025)

In [92]:
df['date'] = pd.to_datetime(df['date'], unit='s', utc=True)   

startDate = pd.to_datetime("2025-08-01", utc=True)
df = df[df['date'] >= startDate]
df

,author,author_flair,title,content,post_id,date,upvotes,upvote_ratio,num_comments,comments,subreddit
0,Sparky-moon,:transpride::AS_Roma:,The Congolese courts have sentenced Jean-Guy B...,"On Tuesday, March 10, 2026, the Congolese cour...",1rpxyqy,2026-03-10 14:10:24+00:00,5,1.00,1,['Some old rich fuck getting put away for misu...,soccer
1,OkayFine101,:Manchester_United:,[@WestHam] West Ham United's dressing room cel...,NaN,1rpxx4c,2026-03-10 14:08:37+00:00,8,1.00,2,[],soccer
2,SolitasTT,:Liverpool:,Eni Aluko wins Joey Barton libel case over pos...,NaN,1rpxsqj,2026-03-10 14:03:41+00:00,11,0.87,5,['Not a good day for Joey Barton. Hopefully ma...,soccer
3,Sparky-moon,:transpride::AS_Roma:,FIFA has cut over $100 million from its operat...,The people who raised the matter of the budget...,1rpxspb,2026-03-10 14:03:38+00:00,3,1.00,1,"[""It wouldn't surprise me if there a comes a d...",soccer
4,AutoModerator,NaN,Daily Discussion,# Welcome to the [r/soccer](https://old.reddit...,1rpxqc2,2026-03-10 14:01:03+00:00,5,1.00,7,['I will riot if we don’t get ucl football nex...,soccer
...,...,...,...,...,...,...,...,...,...,...,...
5095,Claija79,Bot Mexicano 🇲🇽,[Catanzaro] 18th min. Straight away a forced c...,NaN,1qypegy,2026-02-07 21:05:12+00:00,56,0.96,17,"['We’re cursed', 'Lmao. Lmfao even.', 'AC Infi...",ACMilan
5096,Claija79,Bot Mexicano 🇲🇽,El Bebote on IG: “I miss you… but almost there...,NaN,1qyp8yd,2026-02-07 20:59:18+00:00,311,0.98,40,"['6/7 days a week I forget he plays for us', '...",ACMilan
5097,Claija79,Bot Mexicano 🇲🇽,Kevin Zeroli scored for Juve Stabia against Pa...,NaN,1qymv5x,2026-02-07 19:25:05+00:00,77,0.99,4,['https://preview.redd.it/wtj5orbri4ig1.jpeg?w...,ACMilan
5098,Claija79,Bot Mexicano 🇲🇽,Genoa [2] - 2 Napoli - Lorenzo Colombo 57',NaN,1qylee9,2026-02-07 18:29:29+00:00,102,1.00,12,['Good depth and good for lists. \n\nNgl I onl...,ACMilan


get only post content for manual classification

In [93]:
def combine(row):
    comments = row['comments']
    
    if isinstance(comments, list):
        print("LIST")
        return [row['title']] + comments
    elif pd.isna(comments):
        return [row['title']]
    else:  # single string comment
        print(type(comments))
        return [row['title'], comments]

In [94]:
import ast

df['comments'] = df['comments'].apply(
    lambda x: ast.literal_eval(x) if isinstance(x, str) else x
)

In [95]:
temp = df.loc[df['title'] != "Free Talk Friday", ['title', 'comments']].copy()

# combine title and comments
combined = [[title] + comments for title, comments in zip(temp['title'], temp['comments'])]

toClassify = (
    pd.DataFrame({'A': combined})
      .explode('A', ignore_index=True)
      .assign(B=None)
)

In [96]:
# need to fix encoding issue
toClassify['A'] = (
    toClassify['A']
    .str.encode('utf-8')
    .str.decode('utf-8')
)

In [97]:
# send to csv
toClassify.to_csv("classify.csv", index = False, encoding = 'utf-8-sig')